<img src="uva_seal.png">  

## Spark SQL and DataFrames

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: February 22, 2026

---  

### Sources 

Learning Spark, Chapter 9: Spark SQL

https://spark.apache.org/docs/latest/sql-programming-guide.html

https://www.datacamp.com/community/tutorials/apache-spark-tutorial-machine-learning

Demonstration of several useful DataFrame operations:  
https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html

### OBJECTIVES
- Introduction to Spark SQL, the interface for working with structured and semistructured data
- Introduce DataFrames and show basic functionality
- Implement queries using Spark SQL
- Execute tasks including filtering and aggregations using Spark DataFrames
- Review strategies for joining data at scale

### CONCEPTS AND FUNCTIONS
- Spark SQL
- Temp table
- Dataset and DataFrame
- Aggregations with groupBy()
- Joins at scale:
  - Broadcast Hash Join
  - Shuffle Hash Join
  - Sort Merge Join

---  

### 1. Overview

Two important ways of working with big data in Spark: 

- Through Spark SQL
- Using DataFrames

They also interoperate

### 2. SQL in Ten Seconds (tongue in cheek)


SQL is a structured query language used to communicate with relational databases.  
Commands include CREATE, SELECT, UPDATE, ALTER, INSERT INTO, DROP, DELETE.  
This course will use SELECT.

### 3. Spark SQL Capabilities:

- Load data from various structured formats including JSON, Hive, Parquet  
- Query data using SQL inside Spark or from external tools that connect to Spark (e.g., `Tableau`) 
- Spark SQL integrates between SQL and Python/Java/Scala/R code. Can do things like join RDDs and SQL tables.

### 4. Note on Spark SQL Development

- Spark SQL has been heavy development area in new releases 
- As the module involves massive amounts of data, optimizing operations is valuable
 
---

### 5. Dataset and DataFrame

- A Dataset is a distributed collection of data   
- A Dataset can be constructed from JVM objects
- It can be manipulated using functional transformations (`map()`, `flatMap()`, `filter()`, etc.)  
- A DataFrame is a Dataset organized into named columns   

In practice, you will be thinking in terms of `DataFrames`, and not `Datasets`.  

For users familiar with dataframes from R and Python, they are similar, yet with operations distinct to Spark.  

As an example, adding a new column to a DataFrame is executed using `withColumn()`.  

This may feel more formal compared to R and Python.  

Additionally - when compared to R and Python: 

**The Spark DataFrame is built up from RDDs.    
It inherits the properties of distributed data, lazy execution, DAGs.**  

DataFrames can be constructed from a wide array of sources such as:  
structured data files, tables in Hive, external databases, or existing RDDs.  

The DataFrame API is available in Scala, Java, Python, and R. 

---

### 6. DataFrames vs RDDs  

When is it better to use DataFrames, and when is it better to use RDDs?  

Here are some recommendations:   

- In general, most work can be done with DataFrames  

- Use DataFrames to use high-level expressions, to perform SQL queries to explore the data, and to gain columnar access.

**When to use DataFrames**

- If you are thinking about the data by field names, you probably want the data in a DataFrame

- For machine learning and building predictive models, DataFrames are recommended.  
  You will be exploring the data by column, and building features from the columns of data.

**When to use RDDs**
  
- RDDs can be useful to perform low-level transformations and actions on unstructured data.  
  For example, filtering strings and performing other simple transformations on text is best done with RDDs.  
  In these cases, the analyst doesn't care about field names, and there is no need to impose schema on the data.  

- Use RDDs when you want to manipulate the data with functional programming constructs rather than domain specific expressions.

---

### 7. Creating a DataFrame

There are multiple ways to do this:
- use a function such as `read.csv()` to read data from files into DataFrames (most common)
- pass data to `createDataFrame()` with schema
- conversion from RDD using `toDF()`

**Example 1: Create DataFrame from RDD using `toDF()`**

---  
```
# import modules 
from pyspark.sql import Row

# Map the RDD to a DF

df = rdd.map(lambda line: Row(longitude=line[0], 
                              latitude=line[1], 
                              housingMedianAge=line[2],
                              totalRooms=line[3],
                              totalBedRooms=line[4],
                              population=line[5], 
                              households=line[6],
                              medianIncome=line[7],
                              medianHouseValue=line[8])).toDF()
```
---  

**Example 2: Create DataFrame by passing data and schema to `createDataFrame()`**

In [1]:
# import context manager: SparkSession
from pyspark.sql import SparkSession

# import data types
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# set up the session
spark = SparkSession.builder.getOrCreate()

# create some data; list of tuples
data = [
    (0, "ChatGPT is all the rage"),
    (1, "Google released BARD to compete"),
    (2, "What does AWS think about this?")
]

# define schema; each field holds (name, data type, nullable)
# for large number of fields, best to automate schema construction
schema = StructType([StructField('id', IntegerType(), False), 
                     StructField('sentence', StringType(), False)])

# create df by passing data, schema
sentenceDataFrame = spark.createDataFrame(data, schema)

# print first few records
sentenceDataFrame.show(3, False)

# print data type
print(type(sentenceDataFrame))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/22 15:49:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>                                                          (0 + 1) / 1]

+---+-------------------------------+
|id |sentence                       |
+---+-------------------------------+
|0  |ChatGPT is all the rage        |
|1  |Google released BARD to compete|
|2  |What does AWS think about this?|
+---+-------------------------------+

<class 'pyspark.sql.classic.dataframe.DataFrame'>


**Example 3: Create a DataFrame from some JSON data with spark.read()**  
(For an example of JSON data see: http://json.org/example.html)


In [2]:
import os
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Read data in json format
df = spark.read.json("../data/people.json")

# Displays the content of the DataFrame to stdout
df.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



**Creating an RDD from a DataFrame**

This is very simple: `df.rdd`

Here we convert our df containing sentences:

In [3]:
sentence_rdd = sentenceDataFrame.rdd
print(sentence_rdd.take(2))
print(type(sentence_rdd))

[Row(id=0, sentence='ChatGPT is all the rage'), Row(id=1, sentence='Google released BARD to compete')]
<class 'pyspark.core.rdd.RDD'>


---

### 8. Some Useful DataFrames Operations

Next we explore subsetting, filtering, and aggregation among others.

In [4]:
# Read data in json format
df = spark.read.json("../data/people.json")
df.show()

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
|  30|   Andy|
|  19| Justin|
+----+-------+



### Selecting

There are three ways to select columns and we show them all. Find your favorite!

- Bracket operator
- `col()` method
- Dot operator (my favorite)

We see them below

### Filtering

Keep records where age > 21

In [5]:
# bracket operator
df.filter(df['age'] > 21).show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [6]:
# dot operator
df.filter(df.age > 21).show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



In [7]:
# column operator
from pyspark.sql.functions import col

df.filter(col('age') > 21).show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



Keep records subject to filters on name, then sort

In [8]:
from pyspark.sql.functions import asc

# alternatively using df.name instead of col("name")
df.filter((df.name == "Andy") | (df.name == "Michael")).sort(asc("name")).show()

+----+-------+
| age|   name|
+----+-------+
|  30|   Andy|
|NULL|Michael|
+----+-------+



Fetch records with age *null*

In [9]:
df.filter(col("age").isNull()).show() 

+----+-------+
| age|   name|
+----+-------+
|NULL|Michael|
+----+-------+



Fetch records with age *not null*

In [10]:
df.filter(col("age").isNotNull()).show() 

+---+------+
|age|  name|
+---+------+
| 30|  Andy|
| 19|Justin|
+---+------+



### where() is equivalent to filter()

In [11]:
df.where((df.name == "Andy") | (df.name == "Michael")).sort(asc("name")).show()

+----+-------+
| age|   name|
+----+-------+
|  30|   Andy|
|NULL|Michael|
+----+-------+



### Impute missing with 0 (just for illustration; not a great idea for this data)

In [12]:
df.fillna(0).show()

+---+-------+
|age|   name|
+---+-------+
|  0|Michael|
| 30|   Andy|
| 19| Justin|
+---+-------+



### Summarize the age field

In [13]:
df.describe("age").show()

+-------+------------------+
|summary|               age|
+-------+------------------+
|  count|                 2|
|   mean|              24.5|
| stddev|7.7781745930520225|
|    min|                19|
|    max|                30|
+-------+------------------+



---

### 9. Spark SQL Queries

The interface of SQL and DataFrames.    
To write SQL queries against DataFrames, first register DF as a `SQL temp view`, and then write the query.  
A temp view does not persist; it only lasts until the session ends.


**Example of SQL Query against DataFrame**

In [14]:
# register DataFrame as temp view with name "people"
df.createOrReplaceTempView("people")

# query the view
sqlDF = spark.sql("SELECT * FROM people where name == 'Andy'")
sqlDF.show()

+---+----+
|age|name|
+---+----+
| 30|Andy|
+---+----+



---

### 10. Aggregate on columns

SQL functions can be loaded from this library: `pyspark.sql.functions`

We will load some stock data to demonstrate aggregation.

In [15]:
# import data types and functions
from pyspark.sql.types import StructType, StructField, TimestampType, StringType, LongType, FloatType
from pyspark.sql import functions as F

# set up schema
stock_schema = StructType([StructField('date',TimestampType(),False),
                           StructField('ticker',StringType(),False),
                           StructField('close',FloatType(),False),
                           StructField('adjusted_close',FloatType(),False),
                           StructField('volume',LongType(),False),
                         ])

In [16]:
# Read in stock data. Source: Yahoo! finance
DATAPATH_STOCKS = '../data/amzn_msft_prices.csv'

df_stx = spark.read.csv(DATAPATH_STOCKS, header=True, schema=stock_schema)
df_stx.show()

+-------------------+------+------+--------------+--------+
|               date|ticker| close|adjusted_close|  volume|
+-------------------+------+------+--------------+--------+
|2022-02-10 00:00:00|  MSFT|302.38|     299.57297|45386200|
|2022-02-11 00:00:00|  MSFT|295.04|     292.30112|39175600|
|2022-02-14 00:00:00|  MSFT| 295.0|      292.2615|36359500|
|2022-02-15 00:00:00|  MSFT|300.47|      297.6807|27058300|
|2022-02-16 00:00:00|  MSFT| 299.5|     297.33322|29982100|
|2022-02-17 00:00:00|  MSFT|290.73|     288.62668|32461600|
|2022-02-18 00:00:00|  MSFT|287.93|     285.84692|34264000|
|2022-02-22 00:00:00|  MSFT|287.72|     285.63846|41736100|
|2022-02-23 00:00:00|  MSFT|280.27|     278.24234|37811200|
|2022-02-24 00:00:00|  MSFT|294.59|     292.45874|56989700|
|2022-02-25 00:00:00|  MSFT|297.31|     295.15906|32546700|
|2022-02-28 00:00:00|  MSFT|298.79|     296.62836|34627500|
|2022-03-01 00:00:00|  MSFT|294.95|     292.81613|31217800|
|2022-03-02 00:00:00|  MSFT|300.19|     

In [17]:
# check the schema
df_stx.printSchema()

root
 |-- date: timestamp (nullable = true)
 |-- ticker: string (nullable = true)
 |-- close: float (nullable = true)
 |-- adjusted_close: float (nullable = true)
 |-- volume: long (nullable = true)



We will calculate some statistics on each stock:
- minimum closing price
- maximum closing price
- minimum volume
- maximum volume

**IMPORTANT NOTE**  
Do NOT use loops to aggregate data. Loops are run sequentially and do not harness parallelization.  
Using the `groupBy()` method will do the job using parallelization.  
It follows the split, apply, combine method:

- *Split* by one or more grouping variables
- *Apply* a function to the data parts
- *Combine* the result from each grouping level

In [18]:
agg_df = df_stx.groupBy("ticker").agg(F.min("close"), F.max("close"), F.min("volume"), F.max("volume"))
agg_df.show()

+------+----------+----------+-----------+-----------+
|ticker|min(close)|max(close)|min(volume)|max(volume)|
+------+----------+----------+-----------+-----------+
|  AMZN|     81.82|   169.315|   35088600|  272662000|
|  MSFT|    214.25|    315.41|    9200800|   86102000|
+------+----------+----------+-----------+-----------+



---

### 11. Joins at Scale

Joins are one of the most powerful, but also most expensive, operations in PySpark

Spark must:

- Bring matching keys together

- Possibly reshuffle both datasets

- Handle memory pressure

- Avoid skew (to be discussed later)

- Decide the best physical strategy

**Shuffle**

At scale, the cost driver is data movement (shuffle).

Shuffle means:

1 | Data is written to disk

2 | Sent over network

3 | Re-read

4 | Sorted

This slows down jobs and causes a network bottleneck.

Next, we review three join strategies in Spark.

It is best to consider them in order of presentation:  
  > broadcast hash join,  
  > then shuffle hash join,  
  > then sort merge join.


---

**1. Broadcast Hash Join** 

This works great when it's an option. 

Best when one side of join (table) is small.

If one table is small enough, Spark broadcasts it to all executors.

Broadcast data is read only.

Each partition of the large table joins locally, avoiding shuffle.

Example use cases:

- Small dimension table

- Predictors that live in separate table

- Lookup joins


**Code for illustration only:**

```
from pyspark.sql.functions import broadcast

df_large.join(broadcast(df_small), "id")
```

---

**2. Shuffle Hash Join (SHJ)**

Process: 

1 | Shuffle both sides by join key

2 | For each partition:

- Build a hash table on one side

- Probe with the other side

No sorting involved.

Works when:

- No table is small enough to broadcast

- Keys are evenly distributed

**SHJ has a big weakness: It can blow up memory.**

This can happen, for example, if a partition is too large.


---

**3. Sort Merge Join (SMJ); Default for large joins**

This is Spark’s typical strategy for big joins.

Stable and scalable, but expensive.

Used when both datasets to join are large and broadcast is not possible.

Process:

1 | Shuffle both sides

2 | Sort both sides

3 | Merge them


---

### TRY FOR YOURSELF (UNGRADED EXERCISES)

1) Given the stock dataframe, use Spark SQL to select all AMZN records. 

In [ ]:
# SOLUTION

# register stock dataFrame as temp view with name "stocks"
df_stx.createOrReplaceTempView("stocks")

# query the view
sqlDF = spark.sql("SELECT * FROM stocks where ticker == 'AMZN'")
sqlDF.show()

2) Given the stock dataframe, do an aggregation to compute minimum, mean, and maximum adjusted close for each stock.

In [ ]:
df_stx.groupBy("ticker").agg(F.min("adjusted_close"), F.mean("adjusted_close"), F.max("adjusted_close")).show()

3) Select the date, ticker, and adjusted_close columns, saving this data as a parquet file.

In [ ]:
df_stx.select("date","ticker","adjusted_close").write.save("stocks.parquet", format="parquet")

4) Load the parquet file into a new dataframe and verify that things look correct.

In [ ]:
test = spark.read.load("stocks.parquet")
test.show()

---

### 12. Summary

You should now have a basic understanding of:

- Spark SQL
- DataFrames
- When to use DataFrames vs RDDs
- How to use some of the common transformations on DataFrames
- Join strategies at scale

Given practice, you will gain comfort in selecting and processing data with Spark SQL and DataFrames, which is essential.

Additionally, you should have some sense of when DataFrames are preferred over RDDs, and vice versa.
